# Exploración de la ENAHO 2024 para Simula

Este notebook documenta **cómo elegimos y verificamos las variables** que usa Simula, antes de escribir
`scripts/preparar_perfiles.py`. La regla del proyecto: **no asumir ningún nombre ni código**, sino leerlo de los `.dta`.

**Datos:** ENAHO 2024 del INEI (código 966), módulos 01 (vivienda y hogar), 02 (miembros), 03 (educación) y 34 (sumaria).
Los microdatos están en `datos_crudos/`, que **no se sube a GitHub**. Para volver a correr este notebook hay que descargarlos
(ver `CLAUDE.md`). Las salidas solo muestran **conteos y porcentajes**, nunca identificadores de hogares.

**Contenido**
1. Etiquetas de las variables candidatas
2. Encuestas completas e incompletas
3. ¿Dónde se corta urbano/rural?
4. Las 5 preguntas de validación: códigos raros
5. ¿Qué pregunta de internet usar?
6. Comparación con las cifras publicadas por el INEI
7. Perfil del hogar: unión de módulos y jefe(a) de hogar
8. ¿La muestra de 3,000 perfiles representa a la ENAHO?
9. Por qué incluimos las encuestas incompletas

In [1]:
from pathlib import Path

import pandas as pd
import pyreadstat

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CRUDOS = RAIZ / "datos_crudos"
ARCHIVOS = {
    "01": CRUDOS / "mod01/966-Modulo01/enaho01-2024-100.dta",
    "02": CRUDOS / "mod02/966-Modulo02/enaho01-2024-200.dta",
    "03": CRUDOS / "mod03/966-Modulo03/enaho01a-2024-300.dta",
    "34": CRUDOS / "mod34/966-Modulo34/sumaria-2024.dta",
}
LLAVE = ["conglome", "vivienda", "hogar"]
pd.set_option("display.max_colwidth", 90)


def pct_ponderado(df, columna, peso="factor07"):
    """% de hogares con columna == 1, ponderado por el factor de expansión."""
    d = df[df[columna].notna()]
    return 100 * (d[columna] * d[peso]).sum() / d[peso].sum()

## 1. Etiquetas de las variables candidatas

Leemos solo los **metadatos** (sin cargar los datos) para ver el nombre exacto, la etiqueta y los códigos de cada variable.

In [2]:
candidatas = {
    "01": ["ubigeo", "dominio", "estrato", "result", "factor07",
           "p1142", "p1144", "p114b1", "p1143", "p110", "p113a"],
    "02": ["codperso", "p203", "p207", "p208a"],
    "03": ["codperso", "p301a"],
    "34": ["mieperho", "gashog2d", "pobreza", "factor07"],
}
filas = []
for modulo, variables in candidatas.items():
    _, meta = pyreadstat.read_dta(str(ARCHIVOS[modulo]), metadataonly=True)
    for v in variables:
        codigos = meta.variable_value_labels.get(v, {})
        filas.append({"módulo": modulo, "variable": v,
                      "etiqueta": meta.column_names_to_labels.get(v, "NO EXISTE"),
                      "códigos": "; ".join(f"{k}={e}" for k, e in list(codigos.items())[:6])})
pd.DataFrame(filas)

,módulo,variable,etiqueta,códigos
0,01,ubigeo,ubicacion geografica,
1,01,dominio,dominio geografico,1=costa norte; 2=costa centro; 3=costa sur; 4=sierra norte; 5=sierra centro; 6=sierra sur
2,01,estrato,estrato geografico,1= de 500 000 a mas habitantes; 2= de 100 000 a 499 999 habitantes; 3= de 50 000 a 99 ...
3,01,result,resultado final de la encuesta,1=completa; 2=incompleta; 3=rechazo; 4=ausente; 5=vivienda desocupada; 6=no se inicio ...
4,01,factor07,factor de expansion anual. proyecciones cpv-2007,
5,01,p1142,su hogar tiene: telefono celular,0=pase; 1=telefono celular
6,01,p1144,su hogar tiene: conexion a internet,0=pase; 1=conexion a internet
7,01,p114b1,conexion a internet a traves de : conexion fija,0=pase; 1=conexion fija
8,01,p1143,su hogar tiene: conexion a tv por cable o satelital,0=pase; 1=conexion a tv por cable o satelital
9,01,p110,el agua que utilizan en el hogar ¿procede principalmente de:,"1=red publica, dentro de la vivienda; 2=red publica, fuera de la vivienda pero dentro ..."


**Hallazgo:** el diccionario del INEI dice `P1144B1` para el internet fijo, pero en el `.dta` la variable se llama
**`p114b1`**. Si hubiéramos copiado el nombre del diccionario, el script habría fallado.

## 2. Encuestas completas e incompletas

`result` indica el resultado de la entrevista. Usamos las **completas** (`result == 1`) y las **incompletas**
(`result == 2`): en las incompletas el módulo del hogar sí está respondido, y es lo estándar en la ENAHO para variables
del hogar. Al principio usamos solo las completas; la **sección 9** muestra por qué lo cambiamos.

In [3]:
m01, meta01 = pyreadstat.read_dta(
    str(ARCHIVOS["01"]),
    usecols=LLAVE + ["ubigeo", "dominio", "estrato", "result", "factor07", "p1142", "p1144",
                     "p114b1", "p114b2", "p114b3", "p1143", "p110", "p113a", "p1138", "p1139"])
etiquetas_result = meta01.variable_value_labels["result"]
(m01["result"].map(etiquetas_result).value_counts()
 .rename_axis("resultado").to_frame("viviendas"))

,viviendas
resultado,
completa,27623
incompleta,6068
otro,6015
vivienda desocupada,3139
rechazo,1638
ausente,248


In [4]:
hogares = m01[m01["result"].isin([1, 2])].copy()
for c in ["p1142", "p1144", "p114b1", "p114b2", "p114b3", "p1143", "p110", "p113a", "p1138", "p1139"]:
    hogares[c] = pd.to_numeric(hogares[c], errors="coerce")  # algunas vienen como texto
print(f"Hogares con encuesta completa o incompleta: {len(hogares):,}")

Hogares con encuesta completa o incompleta: 33,691


## 3. ¿Dónde se corta urbano/rural?

`estrato` tiene 8 categorías por tamaño del centro poblado. La duda es el **estrato 6** (500 a 1,999 habitantes).
La **Ficha Técnica** de la ENAHO lo resuelve: en el área rural, la unidad de muestreo incluye *"el centro poblado
urbano con 500 a menos de 2 mil habitantes"*. Entonces **urbano = estratos 1 a 5**.

In [5]:
s, _ = pyreadstat.read_dta(str(ARCHIVOS["34"]), usecols=LLAVE + ["mieperho", "gashog2d", "pobreza"])
hogares = hogares.merge(s, on=LLAVE, how="inner")
hogares["poblacion"] = hogares["factor07"] * hogares["mieperho"]  # peso de personas

pd.DataFrame({
    "corte": ["urbano = estratos 1–5 (Ficha Técnica)", "urbano = estratos 1–6"],
    "% población urbana": [100 * hogares.loc[hogares["estrato"] <= k, "poblacion"].sum() / hogares["poblacion"].sum()
                           for k in (5, 6)],
}).round(1)

,corte,% población urbana
0,urbano = estratos 1–5 (Ficha Técnica),80.9
1,urbano = estratos 1–6,83.6


Con el corte de la Ficha Técnica, **80.9%** de la población es urbana, consistente con el ~79% del Censo 2017.

## 4. Las 5 preguntas de validación: códigos raros

En `p113a` (combustible más usado para cocinar) aparecen un **código 9 sin etiqueta** y **569 vacíos**.
Los cruzamos con las preguntas de opción múltiple del mismo bloque.

In [6]:
hogares["p113a"].value_counts(dropna=False).sort_index().rename_axis("p113a").to_frame("hogares")

,hogares
p113a,
1.0,294
2.0,19788
3.0,2786
5.0,269
6.0,6930
7.0,2296
9.0,670
NaN,658


In [7]:
pd.DataFrame({
    "vacíos en p113a con p1138 = 1 ('no cocinan')": [(hogares["p113a"].isna() & (hogares["p1138"] == 1)).sum()],
    "total de vacíos en p113a": [hogares["p113a"].isna().sum()],
    "código 9 en p113a con p1139 = 1 ('bosta, estiércol')": [((hogares["p113a"] == 9) & (hogares["p1139"] == 1)).sum()],
    "total de código 9": [(hogares["p113a"] == 9).sum()],
}).T.rename(columns={0: "hogares"})

,hogares
vacíos en p113a con p1138 = 1 ('no cocinan'),658
total de vacíos en p113a,658
"código 9 en p113a con p1139 = 1 ('bosta, estiércol')",670
total de código 9,670


**Conclusión:** los vacíos son exactamente los hogares que **no cocinan**, y el código 9 corresponde a **bosta o estiércol**
(etiqueta que falta en el diccionario). Para la pregunta *"¿cocinan principalmente con gas?"*: gas = códigos 2 (balón GLP)
y 3 (gas natural); los hogares que no cocinan quedan fuera de esa pregunta.

Para agua, *red pública* = códigos 1 (dentro de la vivienda) y 2 (fuera de la vivienda pero dentro del edificio):

In [8]:
etq = meta01.variable_value_labels["p110"]
(hogares["p110"].map(etq).value_counts(normalize=True).mul(100).round(1)
 .rename_axis("p110: origen del agua").to_frame("% hogares (sin ponderar)"))

,% hogares (sin ponderar)
p110: origen del agua,
"red publica, dentro de la vivienda",80.8
"red publica, fuera de la vivienda pero dentro del edificio",3.3
camion-cisterna u otro similar,3.2
otra,3.2
"rio, acequia, lago, laguna",2.8
manantial o puquio,2.4
pozo (agua subterranea),2.2
pilon o pileta de uso publico,2.1


## 5. ¿Qué pregunta de internet usar?

`p1144` dio **~90%**, mucho más que lo que publica el INEI. El cuestionario lo explica: en 2024 la pregunta es
*"¿Conexión a Internet (fijo/móvil)?"*, y luego `p114b1`–`p114b3` dicen **por qué medio**.
Cuando el hogar no tiene internet, `p114b*` vienen vacías (no se pregunta), así que las tratamos como 0.

In [9]:
for c in ["p114b1", "p114b2", "p114b3"]:
    hogares[c] = hogares[c].fillna(0)
hogares["area"] = hogares["estrato"].le(5).map({True: "Urbano", False: "Rural"})

internet = {
    "p1144 · internet fijo o móvil": "p1144",
    "p114b1 · conexión fija": "p114b1",
    "p114b2 · móvil postpago/control": "p114b2",
    "p114b3 · móvil prepago": "p114b3",
}
tabla = pd.DataFrame({
    "Total": {k: pct_ponderado(hogares, v) for k, v in internet.items()},
    "Lima Metropolitana": {k: pct_ponderado(hogares[hogares["dominio"] == 8], v) for k, v in internet.items()},
    "Urbano": {k: pct_ponderado(hogares[hogares["area"] == "Urbano"], v) for k, v in internet.items()},
    "Rural": {k: pct_ponderado(hogares[hogares["area"] == "Rural"], v) for k, v in internet.items()},
}).round(1)
tabla

,Total,Lima Metropolitana,Urbano,Rural
p1144 · internet fijo o móvil,91.6,95.0,93.9,82.7
p114b1 · conexión fija,27.1,46.8,33.3,3.1
p114b2 · móvil postpago/control,47.2,63.0,54.5,18.8
p114b3 · móvil prepago,59.1,46.1,55.7,72.4


**Decisión:** usamos **`p114b1` (internet fijo)** como pregunta de validación, redactada como *"¿Tu hogar tiene internet
fijo (wifi o cable)?"*. Con `p1144` el 92% responde "sí" y la IA acertaría casi por defecto. `p114b1` en cambio separa
mucho a Lima (47%) del área rural (3%), así que **pone a prueba si la IA entiende las diferencias del Perú**.

## 6. Comparación con las cifras publicadas por el INEI

Referencia: *Informe Técnico Estadísticas de las TIC en los Hogares, IV trimestre 2024* (INEI, marzo 2025).
Nuestros números son del **año completo** 2024, así que no tienen que coincidir exacto.

In [10]:
pd.DataFrame([
    {"indicador": "Hogares con celular, total", "ENAHO 2024 (nuestro)": pct_ponderado(hogares, "p1142"), "INEI IV-2024": 95.1},
    {"indicador": "Hogares con celular, rural", "ENAHO 2024 (nuestro)": pct_ponderado(hogares[hogares["area"] == "Rural"], "p1142"), "INEI IV-2024": 88.0},
    {"indicador": "Hogares con TV cable, rural", "ENAHO 2024 (nuestro)": pct_ponderado(hogares[hogares["area"] == "Rural"], "p1143"), "INEI IV-2024": 10.8},
    {"indicador": "Hogares con internet, total (definición INEI)", "ENAHO 2024 (nuestro)": None, "INEI IV-2024": 58.4},
]).round(1)

,indicador,ENAHO 2024 (nuestro),INEI IV-2024
0,"Hogares con celular, total",95.1,95.1
1,"Hogares con celular, rural",87.5,88.0
2,"Hogares con TV cable, rural",10.3,10.8
3,"Hogares con internet, total (definición INEI)",NaN,58.4


Celular y TV cable coinciden con el INEI (diferencias de menos de 1 punto). El 58.4% de internet del INEI usa una definición
que no reproducimos con estas variables (queda entre el 27% de `p114b1` y el 92% de `p1144`), y no la adivinamos.

## 7. Perfil del hogar: unión de módulos y jefe(a) de hogar

El jefe de hogar es `p203 == 1` en el módulo 02 (sexo `p207`, edad `p208a`); su nivel educativo `p301a` viene del módulo 03.
Verificamos que cada hogar tenga **exactamente un** jefe.

In [11]:
m02, _ = pyreadstat.read_dta(str(ARCHIVOS["02"]), usecols=LLAVE + ["codperso", "p203", "p207", "p208a"])
jefes = m02[pd.to_numeric(m02["p203"], errors="coerce") == 1]
m03, meta03 = pyreadstat.read_dta(str(ARCHIVOS["03"]), usecols=LLAVE + ["codperso", "p301a"])
jefes = jefes.merge(m03, on=LLAVE + ["codperso"], how="left")

jefes_por_hogar = hogares.merge(jefes, on=LLAVE, how="left").groupby(LLAVE).size()
pd.Series({
    "hogares (result 1 y 2)": len(hogares),
    "hogares con exactamente 1 jefe(a)": int((jefes_por_hogar == 1).sum()),
    "jefes sin dato de educación (p301a)": int(jefes.loc[jefes.set_index(LLAVE).index.isin(hogares.set_index(LLAVE).index), "p301a"].isna().sum()),
}).to_frame("conteo")

,conteo
hogares (result 1 y 2),33691
hogares con exactamente 1 jefe(a),33691
jefes sin dato de educación (p301a),5


In [12]:
hogares = hogares.merge(jefes[LLAVE + ["p207", "p208a", "p301a"]], on=LLAVE, how="left")
hogares["gasto_pc_mensual"] = hogares["gashog2d"] / hogares["mieperho"] / 12
etq_pob = {1: "Pobre extremo", 2: "Pobre no extremo", 3: "No pobre"}

perfil = (hogares.assign(pobreza=hogares["pobreza"].map(etq_pob))
          .groupby("area")
          .apply(lambda d: pd.Series({
              "hogares (muestra)": len(d),
              "miembros promedio": (d["mieperho"] * d["factor07"]).sum() / d["factor07"].sum(),
              "gasto per cápita mensual (S/, mediana)": d["gasto_pc_mensual"].median(),
              "% hogares pobres": 100 * d.loc[d["pobreza"] != "No pobre", "factor07"].sum() / d["factor07"].sum(),
              "% jefas de hogar": 100 * d.loc[pd.to_numeric(d["p207"]) == 2, "factor07"].sum() / d["factor07"].sum(),
              "edad promedio del jefe(a)": (pd.to_numeric(d["p208a"]) * d["factor07"]).sum() / d["factor07"].sum(),
          }), include_groups=False)
          .T.round(1))
perfil

area,Rural,Urbano
hogares (muestra),11660.0,22031.0
miembros promedio,3.1,3.4
"gasto per cápita mensual (S/, mediana)",463.9,841.8
% hogares pobres,32.8,19.0
% jefas de hogar,28.6,39.5
edad promedio del jefe(a),55.4,52.9


## 8. ¿La muestra de 3,000 perfiles representa a la ENAHO?

`data/perfiles.parquet` es una muestra aleatoria simple de 3,000 hogares (semilla 42). Como es uniforme, conserva el diseño
de la ENAHO (que sobrerrepresenta lo rural), así que **siempre hay que ponderar por `factor07`**. Comparamos los
porcentajes ponderados:

In [13]:
perfiles = pd.read_parquet(RAIZ / "data" / "perfiles.parquet")
hogares["real_internet_fijo"] = hogares["p114b1"]
hogares["real_gas"] = hogares["p113a"].isin([2, 3]).astype(float).where(hogares["p113a"].notna())
hogares["real_agua_red"] = hogares["p110"].isin([1, 2]).astype(int)

comparacion = pd.DataFrame([
    {"indicador": nombre,
     f"ENAHO 2024 ({len(hogares):,} hogares)": pct_ponderado(hogares, col_enaho),
     "perfiles.parquet (3,000)": pct_ponderado(perfiles, col_perfil)}
    for nombre, col_enaho, col_perfil in [
        ("Celular", "p1142", "real_celular"),
        ("Internet fijo", "real_internet_fijo", "real_internet_fijo"),
        ("TV cable", "p1143", "real_tv_cable"),
        ("Agua de red pública", "real_agua_red", "real_agua_red"),
        ("Cocina con gas", "real_gas", "real_gas"),
    ]])
comparacion["diferencia (pp)"] = comparacion.iloc[:, 2] - comparacion.iloc[:, 1]
comparacion.round(1)

,indicador,"ENAHO 2024 (33,691 hogares)","perfiles.parquet (3,000)",diferencia (pp)
0,Celular,95.1,95.4,0.3
1,Internet fijo,27.1,27.8,0.7
2,TV cable,28.9,30.0,1.1
3,Agua de red pública,86.2,87.3,1.1
4,Cocina con gas,75.8,76.4,0.7


In [14]:
pd.DataFrame({
    "% hogares rurales SIN ponderar": [100 * (perfiles["area"] == "Rural").mean()],
    "% hogares rurales ponderado (factor07)": [100 * perfiles.loc[perfiles["area"] == "Rural", "factor07"].sum() / perfiles["factor07"].sum()],
}).round(1)

,% hogares rurales SIN ponderar,% hogares rurales ponderado (factor07)
0,33.9,20.4


**Conclusiones**
- Las diferencias entre la ENAHO y los 3,000 perfiles son de **poco más de 1 punto como máximo**: la muestra sirve.
- Sin ponderar, los perfiles tienen muchos más hogares rurales de los que hay en el Perú. Por eso la app y la validación
  **sortean hogares con probabilidad proporcional a `factor07`** (`ai/simulador.py`, función `muestrear`).
- Variables finales y su construcción: `scripts/preparar_perfiles.py`. Resultados de la IA contra estas respuestas reales:
  `docs/validacion.md`.

## 9. Por qué incluimos las encuestas incompletas

La primera versión de Simula usaba solo encuestas completas (`result == 1`). Pero las **incompletas** (`result == 2`)
también están en la sumaria con su `factor07`, y **las preguntas del hogar sí están respondidas**. Comparamos los universos:

In [15]:
todos = m01.merge(s, on=LLAVE, how="inner")  # la sumaria trae result 1 y 2
for c in ["p1142", "p1143"]:
    todos[c] = pd.to_numeric(todos[c], errors="coerce")
todos["poblacion"] = todos["factor07"] * todos["mieperho"]
todos["urbano"] = todos["estrato"] <= 5

def resumen(d):
    return pd.Series({
        "hogares en la encuesta": len(d),
        "hogares que representan (suma de factor07)": d["factor07"].sum(),
        "% población urbana": 100 * d.loc[d["urbano"], "poblacion"].sum() / d["poblacion"].sum(),
        "% hogares en Lima Metropolitana (ponderado)": 100 * d.loc[d["dominio"] == 8, "factor07"].sum() / d["factor07"].sum(),
        "% hogares con celular (INEI IV-2024: 95.1)": pct_ponderado(d, "p1142"),
        "% hogares con TV cable": pct_ponderado(d, "p1143"),
    })

pd.DataFrame({
    "solo completas (result = 1)": resumen(todos[todos["result"] == 1]),
    "solo incompletas (result = 2)": resumen(todos[todos["result"] == 2]),
    "completas + incompletas": resumen(todos),
}).round(1)

,solo completas (result = 1),solo incompletas (result = 2),completas + incompletas
hogares en la encuesta,27623.0,6068.0,33691.0
hogares que representan (suma de factor07),7372177.4,2988633.8,10360811.2
% población urbana,74.3,95.6,80.9
% hogares en Lima Metropolitana (ponderado),16.8,71.2,32.5
% hogares con celular (INEI IV-2024: 95.1),94.3,97.1,95.1
% hogares con TV cable,24.5,39.8,28.9


**Conclusión:** las encuestas incompletas son mayoritariamente **urbanas y de Lima** (71% de su peso es Lima Metropolitana).
Al excluirlas, Lima pesaba **16.8%** de los hogares en vez de **32.5%**, y los perfiles quedaban más rurales y menos conectados
que el Perú real (TV cable 24.5% vs 28.9%). Con completas + incompletas, el celular da **95.1%, igual a la cifra del INEI**.

**Decisión:** Simula usa `result` 1 y 2 en `scripts/preparar_perfiles.py`; con eso se regeneraron los perfiles, la
validación de 400 hogares y el ejemplo precargado.